## Run the arbitrage consuming notebook

Make sure that data is flowing into the `crypto.arb.signals` kafka topic:

`kafka-console-consumer.sh   --bootstrap-server localhost:9092   --topic crypto.arb.signals`


# Arbitrage Fee Calculator

This notebook extends the streaming arbitrage project by adding trading fee calculations.
Takes arbitrage signals from the crypto.arb.signals Kafka stream and calculates whether trades would be profitable after accounting for exchange fees.

## Approach
- Reads from Kafka topic crypto.arb.signals
- Applies fee rates from Binance and Coinbase documentation
- Calculates net profit per trade
- Outputs trading recommendations

## Fee Assumptions
Based on exchange documentation (Binance: 0.1%, Coinbase: 0.4% for standard tiers).
These are typical taker fees for retail accounts without VIP status.


## Define Multi-Tier Trading Fees

Here we create a dictionary `FEE_TIERS` which stores trading fee rates for each tier available on Binance and Coinbase. Knowing the fees is crucial to determine if an arbitrage opportunity is actually profitable.
***
**Key Points**
- Fees are based on 2025 Binance and Coinbase documentation)
- Rates are expressed as decimals (e.g., 0.0010 for 0.10%)
- Keys within each tier are:
   - **binance**: binance fee for that tier
   - **coinbase**: coinbase fee for that tier
   - **total_pct**: sum of binance and coinbase fees
   - **description**: Description of the tier and who might use it

In [1]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType

# Fee Tiers Structure (Based on 2025 Binance & Coinbase documentation)
# Sources: Binance VIP Program, Coinbase Advanced Trade tiers
FEE_TIERS = {
    'STANDARD': {
        'binance': 0.0010,    # 0.10% (default retail)
        'coinbase': 0.0050,   # 0.50% average (0.4% maker + 0.6% taker)
        'total_pct': 0.60,
        'description': 'Retail users (default tier)'
    },
    'ACTIVE': {
        'binance': 0.00095,   # 0.095% (VIP 1: 0.09% maker + 0.1% taker avg)
        'coinbase': 0.00325,  # 0.325% (Advanced 1: 0.25% maker + 0.4% taker avg)
        'total_pct': 0.42,
        'description': 'Active traders (≥$1M Binance, ≥$10K Coinbase monthly volume)'
    },
    'INSTITUTIONAL': {
        'binance': 0.00030,   # 0.03% (VIP 5: 0.02% maker + 0.04% taker avg)
        'coinbase': 0.00060,  # 0.06% (Advanced 6: 0.02% maker + 0.1% taker avg)
        'total_pct': 0.09,
        'description': 'Institutional (≥$150M Binance, ≥$50M Coinbase monthly volume)'
    },
    'P2P': {
        'binance': 0.00000,   # 0% (P2P taker fee is 0%)
        'coinbase': 0.00000,   # 0% (Coinbase has no P2P, but for educational purposes we also use 0)
        'total_pct': 0.00,
        'description': 'P2P Trading (Binance P2P only - 0% taker fee)'
    }
}

print("=" * 80)
print("MULTI-TIER FEE MODEL INITIALIZED")
print("=" * 80)
for tier_name, tier_data in FEE_TIERS.items():
    print(f"\n{tier_name}:")
    print(f"  Binance: {tier_data['binance']*100:.3f}%")
    print(f"  Coinbase: {tier_data['coinbase']*100:.3f}%")
    print(f"  Total: {tier_data['total_pct']:.2f}%")
    print(f"  Description: {tier_data['description']}")
print("\n" + "=" * 80)

MULTI-TIER FEE MODEL INITIALIZED

STANDARD:
  Binance: 0.100%
  Coinbase: 0.500%
  Total: 0.60%
  Description: Retail users (default tier)

ACTIVE:
  Binance: 0.095%
  Coinbase: 0.325%
  Total: 0.42%
  Description: Active traders (≥$1M Binance, ≥$10K Coinbase monthly volume)

INSTITUTIONAL:
  Binance: 0.030%
  Coinbase: 0.060%
  Total: 0.09%
  Description: Institutional (≥$150M Binance, ≥$50M Coinbase monthly volume)

P2P:
  Binance: 0.000%
  Coinbase: 0.000%
  Total: 0.00%
  Description: P2P Trading (Binance P2P only - 0% taker fee)



## Configure PySpark for Kafka

Set the PYSPARK_SUBMIT_ARGS environment variable to include the `spark-sql-kafka-0-10` package. This must be set before initializing the spark session to ensure Kafka support is available.

In [2]:
import os

# Set Kafka package before starting Spark
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.4" pyspark-shell'

print("Kafka package configured")

Kafka package configured


## Start Spark Session and Connect to Kafka

1. Initialize the spark session for application "Arbitrage_Fees"
2. Read streaming data from Kafka `crypto.arb.signals` topic
3. Parse incoming JSON messages
4. Define and apply a schema to the parsed JSON data

In [3]:
# Start Spark Session with Kafka support
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Arbitrage_Fees") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0") \
    .getOrCreate()

# Read arbitrage signals from Kafka
signals_raw = (
    spark.readStream
        .format("kafka")
        .option("kafka.bootstrap.servers", "localhost:9092")
        .option("subscribe", "crypto.arb.signals")
        .load()
)

# Parse JSON from Kafka
signals_df = signals_raw.selectExpr("CAST(value AS STRING) as json")

# Define schema
schema = StructType([
    StructField("ts", TimestampType(), True),
    StructField("asset", StringType(), True),
    StructField("direction", StringType(), True),
    StructField("best_spread", DoubleType(), True),
    StructField("spread_buy_binance_sell_coinbase", DoubleType(), True),
    StructField("spread_buy_coinbase_sell_binance", DoubleType(), True),
    StructField("binance_bid", DoubleType(), True),
    StructField("binance_ask", DoubleType(), True),
    StructField("coinbase_bid", DoubleType(), True),
    StructField("coinbase_ask", DoubleType(), True),
])

signals_parsed = signals_df.select(F.from_json("json", schema).alias("d")).select("d.*")

print("Connected to Kafka and reading signals from crypto.arb.signals")


OpenJDK 64-Bit Server VM warning: Unable to get SVE vector length on this system. Disabling SVE. Specify -XX:UseSVE=0 to shun this warning.
OpenJDK 64-Bit Server VM warning: Unable to get SVE vector length on this system. Disabling SVE. Specify -XX:UseSVE=0 to shun this warning.
26/01/17 14:56:51 WARN Utils: Your hostname, osbdet resolves to a loopback address: 127.0.0.1; using 10.0.2.15 instead (on interface enp0s1)
26/01/17 14:56:51 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /home/osbdet/.ivy2/cache
The jars for the packages stored in: /home/osbdet/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-c084469a-f79d-4979-a402-88c46de9f6f8;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.4 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.4 in central
	found org.apache.kafka#kafka-clients;3.4.1 

:: loading settings :: url = jar:file:/home/osbdet/.jupyter_venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.5 in central
	found org.slf4j#slf4j-api;2.0.7 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#commons-pool2;2.11.1 in central
:: resolution report :: resolve 185ms :: artifacts dl 8ms
	:: modules in use:
	com.google.code.findbugs#jsr305;3.0.0 from central in [default]
	commons-logging#commons-logging;1.1.3 from central in [default]
	org.apache.commons#commons-pool2;2.11.1 from central in [default]
	org.apache.hadoop#hadoop-client-api;3.3.4 from central in [default]
	org.apache.hadoop#hadoop-client-runtime;3.3.4 from central in [default]
	org.apache.kafka#kafka-clients;3.4.1 from central in [default]
	org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.4 from central in [default]
	org.ap

Connected to Kafka and reading signals from crypto.arb.signals


## Fee Calculation and Batch Processing

`add_profit_cols()` calculates the outcome of an arbitrage opportunity for a given fee tier.
- Determines buy/sell prices
- Calculates fess
- Calculates net (profit minus total fees)
- Determines an action (`EXECUTE` or `SKIP`)

`process_batch()` is called for each micro batch of data received from Kafka.
- Handles empty batches by skipping processing
- Iterates through the `FEE_TIERS` dictionary and for each tier calls `add_profit_cols()`
- Combines results from all tiers into a single DataFrame
- Filters for `EXECUTE` rows and prints in order of highest `net_profit`

**Note**: `process_batch()` should be passed to `writeStream.foreachBatch()` to handle streaming data.

In [4]:
from pyspark.sql import functions as F

def add_profit_cols(df, fee_binance, fee_coinbase, tier_name):
    # pick buy/sell prices based on direction
    buy_price  = F.when(F.col("direction")=="BUY_BINANCE_SELL_COINBASE", F.col("binance_ask")) \
                  .otherwise(F.col("coinbase_ask"))
    sell_price = F.when(F.col("direction")=="BUY_BINANCE_SELL_COINBASE", F.col("coinbase_bid")) \
                  .otherwise(F.col("binance_bid"))

    # fees depend on which exchange is used on each leg
    buy_fee = F.when(F.col("direction")=="BUY_BINANCE_SELL_COINBASE", buy_price * F.lit(fee_binance)) \
               .otherwise(buy_price * F.lit(fee_coinbase))
    sell_fee = F.when(F.col("direction")=="BUY_BINANCE_SELL_COINBASE", sell_price * F.lit(fee_coinbase)) \
                .otherwise(sell_price * F.lit(fee_binance))

    gross_profit = sell_price - buy_price
    total_fees = buy_fee + sell_fee
    net_profit = gross_profit - total_fees

    # exchanges for display
    buy_ex = F.when(F.col("direction")=="BUY_BINANCE_SELL_COINBASE", F.lit("Binance")).otherwise(F.lit("Coinbase"))
    sell_ex = F.when(F.col("direction")=="BUY_BINANCE_SELL_COINBASE", F.lit("Coinbase")).otherwise(F.lit("Binance"))

    return (df
        .withColumn("tier", F.lit(tier_name))
        .withColumn("buy_exchange", buy_ex)
        .withColumn("sell_exchange", sell_ex)
        .withColumn("buy_price", buy_price)
        .withColumn("sell_price", sell_price)
        .withColumn("gross_profit", gross_profit)
        .withColumn("total_fees", total_fees)
        .withColumn("net_profit", net_profit)
        .withColumn("action", F.when(F.col("net_profit") > 0, F.lit("EXECUTE")).otherwise(F.lit("SKIP")))
    )

def process_batch(batch_df, batch_id):
    if batch_df.isEmpty():
        return

    print(f"\n{'='*100}")
    print(f"BATCH {batch_id} - evaluating opportunities")
    print(f"{'='*100}")

    # Build one DF per tier, then union them
    tier_dfs = []
    for tier_name, fees in FEE_TIERS.items():
        tier_dfs.append(add_profit_cols(batch_df, fees["binance"], fees["coinbase"], tier_name))

    all_tiers = tier_dfs[0]
    for d in tier_dfs[1:]:
        all_tiers = all_tiers.unionByName(d)

    # Only keep profitable rows for printing (and cap!)
    to_print = (all_tiers
        .filter(F.col("action") == "EXECUTE")
        .orderBy(F.col("net_profit").desc())
        .limit(50)  # keep prints manageable
        .collect()
    )

    if not to_print:
        print("No profitable trades in this batch.")
        return

    # Pretty printing (your style)
    for r in to_print:
        print(f"\n{'─'*100}")
        print(f"🔹 [{r['tier']}] ASSET: {r['asset']} | DIRECTION: {r['direction']}")
        print(f"{'─'*100}")
        print(f"  🟢 EXECUTE")
        print(f"     BUY  @ {r['buy_exchange']:8}: ${r['buy_price']:,.2f}")
        print(f"     SELL @ {r['sell_exchange']:8}: ${r['sell_price']:,.2f}")
        print(f"     Gross: ${r['gross_profit']:.4f} | Fees: ${r['total_fees']:.4f}")
        print(f"     NET PROFIT: ${r['net_profit']:.4f}")


# Start Streaming Query

1. Connects to Kafka
2. Applies Batch Processing
3. Uses `outputMode("append")` to ensure only new data that has been processed is considered in each batch
4. Prints confirmation messages, including a summary of the fee tiers

In [5]:
# This connects to Kafka, reads arbitrage signals, and processes them in real-time
query = (
    signals_parsed  # Start with the parsed signals DataFrame from Cell 4
        .writeStream  # Create a streaming query
        .foreachBatch(process_batch)  # Call our function for each batch of data
        .outputMode("append")  # Only process new data (append mode)
        .start()  # Start the query (begins monitoring Kafka)
)

# confirmation messages
print("\n" + "="*100)
print("MULTI-TIER STREAMING QUERY STARTED")
print("="*100)
print("Arbitrage opportunities across 4 fee tiers:")
print("   • STANDARD: Retail (0.60% total fees)")
print("   • ACTIVE: Active traders (0.42% total fees)")
print("   • INSTITUTIONAL: Institutional (0.09% total fees)")
print("   • P2P: Binance P2P (0% total fees)")
print("\n Monitoring crypto.arb.signals Kafka topic...")
print(" To stop: query.stop()")
print("="*100)


MULTI-TIER STREAMING QUERY STARTED
Arbitrage opportunities across 4 fee tiers:
   • STANDARD: Retail (0.60% total fees)
   • ACTIVE: Active traders (0.42% total fees)
   • INSTITUTIONAL: Institutional (0.09% total fees)
   • P2P: Binance P2P (0% total fees)

 Monitoring crypto.arb.signals Kafka topic...
 To stop: query.stop()


26/01/17 14:56:53 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-f93ee45c-b71f-4fe2-adfa-501fe0135f7f. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/01/17 14:56:53 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/01/17 14:56:53 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.



BATCH 1 - evaluating opportunities

────────────────────────────────────────────────────────────────────────────────────────────────────
🔹 [P2P] ASSET: SOL | DIRECTION: BUY_COINBASE_SELL_BINANCE
────────────────────────────────────────────────────────────────────────────────────────────────────
  🟢 EXECUTE
     BUY  @ Coinbase: $143.92
     SELL @ Binance : $143.97
     Gross: $0.0500 | Fees: $0.0000
     NET PROFIT: $0.0500

BATCH 2 - evaluating opportunities

────────────────────────────────────────────────────────────────────────────────────────────────────
🔹 [P2P] ASSET: BTC | DIRECTION: BUY_COINBASE_SELL_BINANCE
────────────────────────────────────────────────────────────────────────────────────────────────────
  🟢 EXECUTE
     BUY  @ Coinbase: $95,378.01
     SELL @ Binance : $95,412.86
     Gross: $34.8500 | Fees: $0.0000
     NET PROFIT: $34.8500

────────────────────────────────────────────────────────────────────────────────────────────────────
🔹 [P2P] ASSET: ETH | DIRECTION


────────────────────────────────────────────────────────────────────────────────────────────────────
🔹 [P2P] ASSET: SOL | DIRECTION: BUY_COINBASE_SELL_BINANCE
────────────────────────────────────────────────────────────────────────────────────────────────────
  🟢 EXECUTE
     BUY  @ Coinbase: $143.92
     SELL @ Binance : $143.97
     Gross: $0.0500 | Fees: $0.0000
     NET PROFIT: $0.0500

────────────────────────────────────────────────────────────────────────────────────────────────────
🔹 [P2P] ASSET: SOL | DIRECTION: BUY_COINBASE_SELL_BINANCE
────────────────────────────────────────────────────────────────────────────────────────────────────
  🟢 EXECUTE
     BUY  @ Coinbase: $143.92
     SELL @ Binance : $143.97
     Gross: $0.0500 | Fees: $0.0000
     NET PROFIT: $0.0500

────────────────────────────────────────────────────────────────────────────────────────────────────
🔹 [P2P] ASSET: XRP | DIRECTION: BUY_COINBASE_SELL_BINANCE
────────────────────────────────────────────────────

26/01/17 14:57:05 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors



BATCH 5 - evaluating opportunities

────────────────────────────────────────────────────────────────────────────────────────────────────
🔹 [P2P] ASSET: BTC | DIRECTION: BUY_COINBASE_SELL_BINANCE
────────────────────────────────────────────────────────────────────────────────────────────────────
  🟢 EXECUTE
     BUY  @ Coinbase: $95,380.75
     SELL @ Binance : $95,412.86
     Gross: $32.1100 | Fees: $0.0000
     NET PROFIT: $32.1100

BATCH 6 - evaluating opportunities

────────────────────────────────────────────────────────────────────────────────────────────────────
🔹 [P2P] ASSET: BTC | DIRECTION: BUY_COINBASE_SELL_BINANCE
────────────────────────────────────────────────────────────────────────────────────────────────────
  🟢 EXECUTE
     BUY  @ Coinbase: $95,378.01
     SELL @ Binance : $95,412.86
     Gross: $34.8500 | Fees: $0.0000
     NET PROFIT: $34.8500

────────────────────────────────────────────────────────────────────────────────────────────────────
🔹 [P2P] ASSET: BTC | D

... (4 outputs left out)


────────────────────────────────────────────────────────────────────────────────────────────────────
🔹 [P2P] ASSET: ETH | DIRECTION: BUY_COINBASE_SELL_BINANCE
────────────────────────────────────────────────────────────────────────────────────────────────────
  🟢 EXECUTE
     BUY  @ Coinbase: $3,302.77
     SELL @ Binance : $3,303.88
     Gross: $1.1100 | Fees: $0.0000
     NET PROFIT: $1.1100

────────────────────────────────────────────────────────────────────────────────────────────────────
🔹 [P2P] ASSET: SOL | DIRECTION: BUY_COINBASE_SELL_BINANCE
────────────────────────────────────────────────────────────────────────────────────────────────────
  🟢 EXECUTE
     BUY  @ Coinbase: $143.94
     SELL @ Binance : $144.01
     Gross: $0.0700 | Fees: $0.0000
     NET PROFIT: $0.0700

────────────────────────────────────────────────────────────────────────────────────────────────────
🔹 [P2P] ASSET: SOL | DIRECTION: BUY_COINBASE_SELL_BINANCE
────────────────────────────────────────────────


────────────────────────────────────────────────────────────────────────────────────────────────────
🔹 [P2P] ASSET: SOL | DIRECTION: BUY_COINBASE_SELL_BINANCE
────────────────────────────────────────────────────────────────────────────────────────────────────
  🟢 EXECUTE
     BUY  @ Coinbase: $143.96
     SELL @ Binance : $144.01
     Gross: $0.0500 | Fees: $0.0000
     NET PROFIT: $0.0500

BATCH 23 - evaluating opportunities

────────────────────────────────────────────────────────────────────────────────────────────────────
🔹 [P2P] ASSET: ETH | DIRECTION: BUY_COINBASE_SELL_BINANCE
────────────────────────────────────────────────────────────────────────────────────────────────────
  🟢 EXECUTE
     BUY  @ Coinbase: $3,302.77
     SELL @ Binance : $3,303.88
     Gross: $1.1100 | Fees: $0.0000
     NET PROFIT: $1.1100

BATCH 24 - evaluating opportunities

────────────────────────────────────────────────────────────────────────────────────────────────────
🔹 [P2P] ASSET: BTC | DIRECTION: 


────────────────────────────────────────────────────────────────────────────────────────────────────
🔹 [P2P] ASSET: BTC | DIRECTION: BUY_COINBASE_SELL_BINANCE
────────────────────────────────────────────────────────────────────────────────────────────────────
  🟢 EXECUTE
     BUY  @ Coinbase: $95,378.81
     SELL @ Binance : $95,412.86
     Gross: $34.0500 | Fees: $0.0000
     NET PROFIT: $34.0500

────────────────────────────────────────────────────────────────────────────────────────────────────
🔹 [P2P] ASSET: ETH | DIRECTION: BUY_COINBASE_SELL_BINANCE
────────────────────────────────────────────────────────────────────────────────────────────────────
  🟢 EXECUTE
     BUY  @ Coinbase: $3,302.77
     SELL @ Binance : $3,303.88
     Gross: $1.1100 | Fees: $0.0000
     NET PROFIT: $1.1100

BATCH 27 - evaluating opportunities

────────────────────────────────────────────────────────────────────────────────────────────────────
🔹 [P2P] ASSET: ETH | DIRECTION: BUY_COINBASE_SELL_BINANCE
───

26/01/17 14:57:48 ERROR Executor: Exception in task 0.0 in stage 63.0 (TID 156)
java.lang.IllegalStateException: Pool not open
	at org.apache.commons.pool2.impl.BaseGenericObjectPool.assertOpen(BaseGenericObjectPool.java:440)
	at org.apache.commons.pool2.impl.GenericKeyedObjectPool.borrowObject(GenericKeyedObjectPool.java:416)
	at org.apache.commons.pool2.impl.GenericKeyedObjectPool.borrowObject(GenericKeyedObjectPool.java:350)
	at org.apache.spark.sql.kafka010.consumer.InternalKafkaConsumerPool.borrowObject(InternalKafkaConsumerPool.scala:85)
	at org.apache.spark.sql.kafka010.consumer.KafkaDataConsumer.$anonfun$retrieveConsumer$1(KafkaDataConsumer.scala:604)
	at org.apache.spark.sql.kafka010.consumer.KafkaDataConsumer.timeNanos(KafkaDataConsumer.scala:666)
	at org.apache.spark.sql.kafka010.consumer.KafkaDataConsumer.retrieveConsumer(KafkaDataConsumer.scala:604)
	at org.apache.spark.sql.kafka010.consumer.KafkaDataConsumer.getOrRetrieveConsumer(KafkaDataConsumer.scala:588)
	at org.apach